# PointFusion3D: Build Your Own 3D Scan Registration Pipeline
### Reboot × Spatial AI Foundations — a 4-session hands-on intro to 3D Computer Vision

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dwaynekj/3d-portfolio-starter/blob/main/PointFusion3D_student_FULL.ipynb)

**Instructor:** Dwayne Jones

Welcome! Over these four sessions you're going to build a real Computer Vision pipeline from
scratch: something that takes two separate, messy, partial 3D scans of the same object and
**fuses them into one clean 3D model** — a task called *point cloud registration*. It's one of
the actual building blocks behind robotics, AR/VR, autonomous vehicles, and 3D scanning apps.

You'll fill in a handful of `# TODO` sections using the same kind of thinking tested in
technical interviews (loops, distance calculations, convergence checks) — but applied to real
3D data instead of abstract puzzles. By the end you'll have a working, GitHub-ready portfolio
project **and** a resume bullet + interview story to go with it.

**Session map**

| Session | What you'll do in this notebook |
|---|---|
| S1 — Why Spatial AI | (no code — career context, see slides) |
| S2 — Seeing in 3D | Parts 1-3: load a point cloud, clean messy data, implement nearest-neighbor search |
| S3 — From Data to Models | Parts 4-7: implement the ICP loop, validate on real scan data, bridge to a 3D Gaussian Splat |
| S4 — Your Next Step | Parts 8-9: (optional) run it on a scan of your own, then ship your portfolio piece |
| Homework (after S2) | Turn a real phone scan (Elmo) into a point cloud by writing your own surface sampler |

Run cells top to bottom. Cells marked **`# TODO`** are yours to complete — everything else is
provided so you can focus on the algorithmic core, not boilerplate.

## Setup
Run this once per Colab session. Takes about 30-60 seconds. It installs Open3D and, if the data
files aren't already next to this notebook, downloads them from the course GitHub repo.

In [ ]:
!pip install -q open3d
import os
import subprocess

# Get the data files. In Colab the notebook starts with an empty folder, so clone the repo
# and work from inside it. If you cloned the repo yourself, the data is already here.
if not os.path.exists("data/scanA_bunny_partial1.ply"):
    if not os.path.isdir("3d-portfolio-starter"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/Dwaynekj/3d-portfolio-starter.git"], check=True)
    os.chdir("3d-portfolio-starter")
print("Data files:", sorted(os.listdir("data")))

import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D projection)

print("Open3D version:", o3d.__version__)

## Part 1 — What *is* a point cloud?

A point cloud is just a big array of `(x, y, z)` coordinates — one row per point a sensor
measured on a surface. That's it. No mesh, no faces, just a "cloud" of 3D samples. A LiDAR
sensor, a depth camera, or a phone's photogrammetry app (like the KIRI Engine demo — more in
Part 8) all produce one of these as their raw output.

Our object is the classic **Stanford Bunny**. On the left is what the real object looks like as a
solid 3D model (a mesh: a surface built from triangles). On the right is what a 3D scanner actually
hands you: just points sampled from that surface, with no triangles connecting them.

![The Stanford Bunny as a mesh and as a point cloud](https://raw.githubusercontent.com/Dwaynekj/3d-portfolio-starter/main/assets/stanford_bunny_mesh_vs_points.png)

Let's load the scan on the right and look at it as a plain NumPy array before we do anything
fancy with it.

In [ ]:
# Given: small helper to plot a point cloud (or two) in 3D. You'll reuse this a lot.
def plot_clouds(clouds, colors, labels, title="", s=2):
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")
    for pts, c, lab in zip(clouds, colors, labels):
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=c, s=s, alpha=0.6, label=lab)
    all_pts = np.vstack(clouds)
    center = all_pts.mean(axis=0)
    radius = (all_pts.max(axis=0) - all_pts.min(axis=0)).max() / 2 * 1.1
    ax.set_xlim(center[0]-radius, center[0]+radius)
    ax.set_ylim(center[1]-radius, center[1]+radius)
    ax.set_zlim(center[2]-radius, center[2]+radius)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_title(title)
    ax.legend()
    plt.show()

scan_a = o3d.io.read_point_cloud("data/scanA_bunny_partial1.ply")
pts_a = np.asarray(scan_a.points)
print("scan_a is just a NumPy array under the hood:", type(pts_a), pts_a.shape)
print("first 3 points (x, y, z):\n", pts_a[:3])

plot_clouds([pts_a], ["#3b82f6"], ["Scan A"], title=f"Scan A - {len(pts_a)} points")

## Part 2 — Real data is messy: filtering outliers

Now load **Scan B** — a second, independent scan of the same bunny from a different angle. In
the real world this would be a second pass of a phone scan, or a second LiDAR sweep. It's been
deliberately roughed up the way real sensor data actually looks:

- it's **rotated and shifted** relative to Scan A (different viewpoint)
- it's **noisy** (sensor measurement error)
- it's **partial** (the sensor didn't see the whole object)
- it has **outlier junk points** (reflections, sensor glitches — every real scan has these)

Plot it and you'll see the problem immediately.

In [ ]:
scan_b_raw = o3d.io.read_point_cloud("data/scanB_bunny_partial2_raw.ply")
pts_b_raw = np.asarray(scan_b_raw.points)
plot_clouds([pts_a, pts_b_raw], ["#3b82f6", "#f97316"], ["Scan A", "Scan B (raw)"],
            title="Two independent scans of the same object")

### TODO 1: write your own outlier filter

Before we can match these two scans up, we need to throw out the junk points in Scan B. The
idea: **a real surface point has other points nearby; an outlier sits alone.**

You're given a KD-tree (a data structure for fast nearest-neighbor lookups — think of it like a
structure that lets you ask "what's near this point?" without checking every other point one by
one). Your job:

1. For every point, find the distance to its `k` nearest neighbors (excluding itself).
2. Compute the **mean** of those `k` distances — this is the point's "local density" score. A
   point buried inside the surface will have small, consistent neighbor distances. An outlier
   sitting off on its own will have a much larger mean distance.
3. A point is kept if its mean neighbor distance is **below** `global_mean + std_ratio * global_std`
   (i.e. it isn't a statistical outlier compared to the rest of the cloud).

Return a boolean NumPy array (`True` = keep this point).

In [ ]:
def remove_outliers(points, k=20, std_ratio=2.0):
    # points: (N, 3) array
    # k: number of nearest neighbors to check per point
    # std_ratio: how many standard deviations above the mean counts as an outlier
    # returns: (N,) boolean array, True = keep the point
    tree = o3d.geometry.KDTreeFlann(o3d.geometry.PointCloud(points=o3d.utility.Vector3dVector(points)))
    mean_dists = np.zeros(len(points))

    # TODO: for each point, use tree.search_knn_vector_3d(point, k+1) to find its k nearest
    # neighbors (the +1 accounts for the point finding itself at distance 0 -- skip index 0).
    # search_knn_vector_3d returns (num_found, indices, squared_distances).
    # Fill in mean_dists[i] with the mean of the sqrt of those k squared distances.
    for i, p in enumerate(points):
        raise NotImplementedError("compute mean_dists[i] here")

    global_mean = mean_dists.mean()
    global_std = mean_dists.std()
    keep_mask = mean_dists < (global_mean + std_ratio * global_std)
    return keep_mask

keep = remove_outliers(pts_b_raw)
pts_b_clean = pts_b_raw[keep]
print(f"outlier removal: {len(pts_b_raw)} -> {len(pts_b_clean)} points ({(~keep).sum()} removed)")
plot_clouds([pts_a, pts_b_clean], ["#3b82f6", "#f97316"], ["Scan A", "Scan B (cleaned)"],
            title="After outlier removal")

## Part 3 — Nearest-neighbor correspondence search

To align two point clouds, we first need to guess **which point in Scan A corresponds to which
point in Scan B**. We don't have labels telling us this — we have to search for it.

The simplest, surprisingly effective approach: **for every point in A, its correspondence in B
is just its nearest neighbor in B.** Some of these guesses will be wrong (especially early on,
before the scans are aligned) — that's expected and we'll deal with it in Part 4.

This is the "nearest-neighbor search" building block — the same core idea as a k-NN classifier
or a k-NN lookup in a database, just applied to 3D coordinates instead of feature vectors.

In [ ]:
def find_correspondences(src_points, tgt_points):
    # For every point in src_points, find the index of its nearest point in tgt_points.
    # src_points: (N, 3) array
    # tgt_points: (M, 3) array
    # returns:
    #   indices: (N,) int array -- indices[i] is the row in tgt_points closest to src_points[i]
    #   distances: (N,) float array -- the corresponding distance
    tgt_cloud = o3d.geometry.PointCloud(points=o3d.utility.Vector3dVector(tgt_points))
    tgt_tree = o3d.geometry.KDTreeFlann(tgt_cloud)

    indices = np.zeros(len(src_points), dtype=np.int64)
    distances = np.zeros(len(src_points))

    # TODO: for each point p in src_points, call tgt_tree.search_knn_vector_3d(p, 1) to get
    # the single nearest neighbor in tgt_points. It returns (num_found, [index], [sq_distance]).
    # Store the index in indices[i] and sqrt(sq_distance) in distances[i].
    for i, p in enumerate(src_points):
        raise NotImplementedError("fill in indices[i] and distances[i]")

    return indices, distances

# quick sanity check on a tiny toy example you can verify by eye
toy_src = np.array([[0., 0., 0.], [5., 5., 5.]])
toy_tgt = np.array([[0.1, 0., 0.], [10., 10., 10.], [5.2, 4.9, 5.0]])
idx, dist = find_correspondences(toy_src, toy_tgt)
print("matched indices:", idx, " (expected [0 2])")
print("distances:", dist)
assert list(idx) == [0, 2], "toy_src[0] should match toy_tgt[0], toy_src[1] should match toy_tgt[2]"
print("PASS: toy example matched correctly")

## Part 4 — The ICP loop (Iterative Closest Point)

Here's the full algorithm you're implementing, in plain English:

1. Guess correspondences (Part 3 — you just built this).
2. Given those correspondences, compute the best rigid rotation + translation that lines them
   up (this is a small linear-algebra step called Procrustes/Kabsch alignment — **provided for
   you below**, since it's more linear algebra than algorithmic thinking).
3. Apply that transform, moving the source cloud closer to the target.
4. **Repeat** — because step 1's guesses were based on the *old*, less-aligned positions, they
   get better each time you redo them on the newly-moved cloud.
5. Stop when the alignment error stops improving (this is a **convergence check** — the same
   "keep looping until the value stabilizes" pattern you'd use in almost any iterative algorithm).

This loop — guess, fit, move, repeat, check for convergence — is Iterative Closest Point (ICP),
one of the most widely used algorithms in robotics and 3D vision.

In [ ]:
# Given: least-squares rigid rotation + translation that maps points A onto points B.
def best_fit_transform(A, B):
    centroid_A = A.mean(axis=0)
    centroid_B = B.mean(axis=0)
    AA = A - centroid_A
    BB = B - centroid_B
    H = AA.T @ BB
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:  # reflection case
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = centroid_B - R @ centroid_A
    return R, t

### TODO 2: complete the ICP loop

Fill in the loop body. On each iteration you need to:
1. Call your `find_correspondences` to match the (currently moved) source points to the target.
2. Reject bad matches — keep only correspondences with `distance < dist_threshold` (this is a
   simple but important outlier-rejection step: early on, plenty of guessed matches will be
   nonsense, and letting them influence the fit slows convergence).
3. Call `best_fit_transform` on the surviving matched pairs to get `R, t`.
4. Apply the transform to `src_points` (`src_points = (R @ src_points.T).T + t`).
5. Check for convergence: if the mean matched distance barely changed from the previous
   iteration (less than `tolerance`), stop early.

In [ ]:
def icp(src_points, tgt_points, max_iterations=40, tolerance=1e-6, dist_threshold=0.02):
    src_points = src_points.copy()
    prev_error = None

    for iteration in range(max_iterations):
        # TODO:
        # 1) idx, dist = find_correspondences(src_points, tgt_points)
        # 2) keep = dist < dist_threshold
        # 3) R, t = best_fit_transform(src_points[keep], tgt_points[idx[keep]])
        # 4) src_points = (R @ src_points.T).T + t
        # 5) mean_error = dist[keep].mean()
        # 6) if prev_error is not None and abs(prev_error - mean_error) < tolerance: break
        # 7) prev_error = mean_error
        raise NotImplementedError("implement one ICP iteration per the steps above")

    return src_points, prev_error

aligned_b, final_err = icp(pts_b_clean, pts_a)
print(f"final mean correspondence error: {final_err:.5f}")
plot_clouds([pts_a, aligned_b], ["#3b82f6", "#f97316"], ["Scan A", "Scan B (aligned)"],
            title="After ICP registration")

## Part 5 — Grade yourself: does it actually work?

Because we generated Scan B by applying a **known** rotation and translation to Scan A, we can
check your ICP implementation against ground truth — no instructor needed. This is the same
"write a test that proves it" habit that separates a working portfolio project from "I think it
works."

If this cell prints **PASS**, your registration pipeline is correct end-to-end.

In [ ]:
R_true = np.load("data/bunny_known_R.npy")
t_true = np.load("data/bunny_known_t.npy")
# Scan B was generated as B = R_true @ A + t_true. Your ICP aligns B onto A (the reverse
# direction), so the transform it recovers should match the INVERSE of the known transform.
R_true_inv = R_true.T
t_true_inv = -R_true.T @ t_true

# recover the ESTIMATED transform by fitting clean-B -> aligned-B (i.e. what ICP applied)
R_est, t_est = best_fit_transform(pts_b_clean, aligned_b)

rotation_error_deg = np.degrees(np.arccos(np.clip((np.trace(R_true_inv.T @ R_est) - 1) / 2, -1, 1)))
translation_error = np.linalg.norm(t_true_inv - t_est)

print(f"rotation error:    {rotation_error_deg:.3f} degrees")
print(f"translation error: {translation_error:.5f}")

assert rotation_error_deg < 5.0, "Rotation error too high -- check your ICP loop"
assert translation_error < 0.02, "Translation error too high -- check your ICP loop"
print("\nPASS - your ICP pipeline recovered the known transform within tolerance.")

## Part 6 — Real-world validation: does it work on messy, real sensor data?

The bunny pair was synthetic (so we could grade it). Now let's run your **exact same functions**
— no changes — on two real fragments from an indoor RGB-D scan of a room (the Redwood dataset,
the same pair Open3D ships as `DemoICPPointClouds`). This is genuine sensor data with real noise
and only partial overlap between the two views — the same situation you'd face in an actual job.

Two things are different from the bunny:

- **Coarse-to-fine.** Real scans only partly overlap, so one ICP run with a single threshold tends
  to lock onto a wrong-but-plausible fit. Instead we run your `icp()` twice: first with a loose
  `dist_threshold` to pull the scans roughly together, then with a tight one to lock them in.
  This is how ICP is used in practice, and a good thing to mention in an interview.
- **We still know the answer.** The true alignment of these two fragments was computed once with
  Open3D's reference ICP and saved as `data/room_A_to_B.npy`, so this part grades itself too.

If your Part 2-4 code is general (not hard-coded to the bunny), this should just work.

In [ ]:
# Two fragments of a real room scan (Redwood RGB-D), downsampled to 5 cm so ICP runs in seconds.
room_a = o3d.io.read_point_cloud("data/room_fragment_A_downsampled.ply").voxel_down_sample(0.05)
room_b = o3d.io.read_point_cloud("data/room_fragment_B_downsampled.ply").voxel_down_sample(0.05)
room_a_pts = np.asarray(room_a.points)
room_b_pts = np.asarray(room_b.points)

# The two fragments were captured from different camera positions. T_true is the known 4x4
# transform that lines fragment A up with fragment B.
T_true = np.load("data/room_A_to_B.npy")
room_a_true = (T_true[:3, :3] @ room_a_pts.T).T + T_true[:3, 3]

# Nudge fragment A out of alignment (rotate 15 degrees around its center and shift it
# 30 cm / 20 cm) so there's something visible for your ICP to fix.
theta = np.deg2rad(15)
R_perturb = np.array([[np.cos(theta), -np.sin(theta), 0],
                      [np.sin(theta),  np.cos(theta), 0],
                      [0, 0, 1]])
center = room_a_true.mean(axis=0)
room_a_offset = (R_perturb @ (room_a_true - center).T).T + center + np.array([0.3, 0.2, 0.0])

plot_clouds([room_a_offset, room_b_pts], ["#3b82f6", "#f97316"], ["Fragment A", "Fragment B"],
            title="Real scan data -- before registration", s=1.5)

# Coarse-to-fine: a loose threshold pulls the scans together, a tight one locks them in.
coarse_room_a, _ = icp(room_a_offset, room_b_pts, dist_threshold=0.3, max_iterations=100)
aligned_room_a, room_err = icp(coarse_room_a, room_b_pts, dist_threshold=0.1, max_iterations=100)
print(f"final mean correspondence error on real data: {room_err:.4f} m")

# Grade it: how far is every point of your aligned fragment from where it truly belongs?
pose_error = np.linalg.norm(aligned_room_a - room_a_true, axis=1).mean()
print(f"average distance from the true pose: {pose_error * 100:.1f} cm")
assert pose_error < 0.05, "More than 5 cm off the true pose -- check your ICP loop"
print("PASS - your ICP aligned real sensor data to within a few centimeters.")

plot_clouds([room_b_pts, aligned_room_a], ["#f97316", "#3b82f6"], ["Fragment B", "Fragment A (aligned)"],
            title="Real scan data -- after your ICP", s=1.5)

## Part 7 — Bridging to a model: from point cloud to Gaussian Splat

In the live session, we'll demo a **3D Gaussian Splat** — a technique that renders a photoreal
3D scene from a set of "soft," colored, blurry 3D blobs (Gaussians) instead of a mesh. Training
one for real takes a dedicated GPU pipeline (there's a link to try it yourself at the bottom of
this section) — but the *starting point* for that training is exactly the kind of point cloud
you just built.

Here's the small, concrete piece you can actually build right now: turning your registered point
cloud into the **initial seed** a Gaussian Splat trainer would start from — one Gaussian per
point, positioned at that point, with a starting size based on how dense the cloud is nearby.

In [ ]:
def make_gaussian_seed(points, k=8):
    # Convert a point cloud into an initial (very rough) Gaussian-splat seed:
    # for each point, estimate a starting "size" (scale) from the average distance
    # to its k nearest neighbors -- denser regions get smaller initial Gaussians,
    # sparser regions get bigger ones.
    #
    # returns a dict with 'positions' (N,3) and 'scales' (N,) -- what a real trainer
    # (e.g. gsplat / nerfstudio's splatfacto) would take as its starting point.
    cloud = o3d.geometry.PointCloud(points=o3d.utility.Vector3dVector(points))
    tree = o3d.geometry.KDTreeFlann(cloud)
    scales = np.zeros(len(points))

    # TODO: for each point, find its k nearest neighbors (excluding itself, like Part 2)
    # and set scales[i] to the mean neighbor distance. That mean distance becomes the
    # Gaussian's initial radius -- roughly "how far to the next point around here."
    for i, p in enumerate(points):
        raise NotImplementedError("compute scales[i] from the mean k-NN distance")

    return {"positions": points, "scales": scales}

seed = make_gaussian_seed(aligned_room_a[:500])  # small subset just to keep this instant
print("seed positions shape:", seed["positions"].shape)
print("seed scale range:", seed["scales"].min(), "-", seed["scales"].max())
print("\nThat's literally the init step of a real Gaussian-splat trainer.")
print("Want to actually train one? Try:")
print("https://colab.research.google.com/github/camenduru/gaussian-splatting-colab/blob/main/gaussian_splatting_colab.ipynb")
print("(needs a T4 GPU runtime in Colab -- Runtime > Change runtime type -- and takes ~10-20 min per scene)")

## Part 8 — (Optional, after the workshop) Bring your own scan

Everything above ran on pre-packaged data. If you want to push this further on your own time —
**totally optional, not required to finish the program** — here's how to run your exact same
pipeline on an object you scan yourself, with just your phone:

1. **Install KIRI Engine** (free, Android or iOS, no LiDAR/Pro phone needed — it uses regular
   photogrammetry) — search "KIRI Engine" in the App Store / Play Store.
2. Pick a small object (a mug, a shoe, a water bottle).
3. Open **Photo Scan** mode and capture the **front half** of the object as one scan — walk
   around it, following the app's on-screen guide, ~30-80 photos.
4. Export that scan as a **.ply** file (Export → PLY). Rename it `my_scan_A.ply`.
5. Do a second Photo Scan pass capturing the **back half** of the same object. Export as
   `my_scan_B.ply`.
   - The free tier gives you unlimited scanning and processing, and at least 3 exports per
     week -- plenty for this.
6. Upload both `.ply` files into this Colab session (folder icon on the left → upload), or into
   a Google Drive folder and mount your Drive (code below).
7. Re-run Parts 2-6 above, swapping in `my_scan_A.ply` / `my_scan_B.ply` for the bunny files.

Two slightly different angles of the same object works just as well as a strict front/back split
-- that's exactly the "partial views that need fusing" problem this whole notebook solves.

In [ ]:
# Uncomment and run this if you're uploading your scans via Google Drive instead of
# Colab's file-upload button:
#
# from google.colab import drive
# drive.mount('/content/drive')
# my_scan_a = o3d.io.read_point_cloud('/content/drive/MyDrive/my_scan_A.ply')
# my_scan_b = o3d.io.read_point_cloud('/content/drive/MyDrive/my_scan_B.ply')
print("See the markdown above for the KIRI Engine capture steps.")

## Part 9 — Ship it: package this as a portfolio piece

You now have a real, working, self-checking Computer Vision pipeline. Here's how to turn it into
something a hiring manager actually sees:

1. **Save this notebook to GitHub directly from Colab**: File → Save a copy in GitHub. Name the
   repo something specific, e.g. `point-cloud-registration` — not `spatial-ai-workshop`.
2. **Write a short README** with: what the project does (1-2 sentences), the before/after image
   from Part 4 or Part 6, and how to run it.
3. **Resume bullet template:**
   > Built a 3D point-cloud registration pipeline in Python/Open3D, implementing nearest-neighbor
   > correspondence search and an Iterative Closest Point (ICP) alignment loop to fuse multi-view
   > 3D scans; validated against known ground truth and real-world RGB-D scan data.
4. **Interview talking point:** be ready to explain, in your own words, why you reject
   correspondences past a distance threshold (Part 4), and what would happen if you skipped that
   step (hint: try setting `dist_threshold` very high and see what happens to your alignment!).

### Keep learning
- Open3D tutorials: http://www.open3d.org/docs/release/tutorial/pipelines/icp_registration.html
- Gaussian Splatting Colab (go further on Part 7): https://colab.research.google.com/github/camenduru/gaussian-splatting-colab/blob/main/gaussian_splatting_colab.ipynb
- KIRI Engine (capture your own scans): https://www.kiriengine.app/

Nice work — this is a real project. Go show it off.

## Homework — Turn a phone scan into a point cloud (do after Session 2)

In Session 2 you saw KIRI Engine turn phone photos of an Elmo toy into a 3D model. That model is
a **mesh**: a surface made of triangles (about 100,000 of them), not a point cloud. Everything in
this notebook works on point clouds, so the first step with any real scan is to **sample points
from its surface**. You'll write that sampler yourself.

Run **Setup** and **Part 1** first (you need `np`, `o3d` and `plot_clouds`). Then run the cell below
to download the Elmo scan. It's kept out of the repo because it's 11 MB, and only this homework
needs it.

In [ ]:
# Given: download and unzip the Elmo scan from the repo's GitHub Release (runs once).
import urllib.request
import zipfile

ELMO_URL = "https://github.com/Dwaynekj/3d-portfolio-starter/releases/download/data-elmo-v1/elmo_scan.zip"
ELMO_DIR = "data/kiri/elmo_scan"
ELMO_OBJ = f"{ELMO_DIR}/3DModel.obj"

if not os.path.exists(ELMO_OBJ):
    os.makedirs(ELMO_DIR, exist_ok=True)
    urllib.request.urlretrieve(ELMO_URL, "data/kiri/elmo_scan.zip")
    with zipfile.ZipFile("data/kiri/elmo_scan.zip") as z:
        z.extractall(ELMO_DIR)  # the zip holds 3DModel.obj, .mtl and .jpg

o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)  # hide texture warnings
elmo_mesh = o3d.io.read_triangle_mesh(ELMO_OBJ)
elmo_vertices = np.asarray(elmo_mesh.vertices)    # (V, 3) corner positions
elmo_triangles = np.asarray(elmo_mesh.triangles)  # (T, 3) each row = 3 row-indices into elmo_vertices
print(f"Elmo mesh: {len(elmo_vertices)} vertices, {len(elmo_triangles)} triangles")

### HW TODO: sample points evenly from a mesh surface

Your function gets the mesh as two arrays and returns `n_points` random points spread **evenly over
the surface**. Write it with NumPy array operations, no Python `for` loops (vectorized ops are a
big part of why 3D code runs fast).

1. **Triangle areas.** For a triangle with corners `A, B, C`, the area is
   `0.5 * |cross(B - A, C - A)|`. Compute all of them at once with `np.cross` and
   `np.linalg.norm(..., axis=1)`.
2. **Pick triangles, weighted by area.** A triangle twice as big should get twice as many points.
   Use `rng.choice(num_triangles, size=n_points, p=...)`, where `p` is each triangle's share of the
   total area. If you pick triangles uniformly instead, small, detailed regions (like Elmo's face)
   get far too many points.
3. **Pick a random spot inside each chosen triangle.** Draw `r1 = np.sqrt(rng.random(n))` and
   `r2 = rng.random(n)`, then the point is `(1 - r1) * A + r1 * (1 - r2) * B + r1 * r2 * C`.
   This is the standard formula for a uniform point in a triangle. The `sqrt` matters: without it,
   points bunch up near corner `A`.

Hints: `A = vertices[triangles[:, 0]]` gives the first corner of *every* triangle at once, shape
`(T, 3)`. To use the `(n,)` arrays `r1` and `r2` with `(n, 3)` corner arrays, add an axis:
`r1[:, None]`.

In [ ]:
def sample_points_on_mesh(vertices, triangles, n_points, seed=0):
    # vertices: (V, 3) array of corner positions
    # triangles: (T, 3) int array; each row holds 3 row-indices into vertices
    # returns: (n_points, 3) array of points spread evenly over the surface
    rng = np.random.default_rng(seed)
    A = vertices[triangles[:, 0]]  # (T, 3) first corner of every triangle
    B = vertices[triangles[:, 1]]
    C = vertices[triangles[:, 2]]

    # TODO 1: areas = (T,) array of triangle areas
    # TODO 2: chosen = (n_points,) triangle indices, picked with probability proportional to area
    # TODO 3: r1, r2 and the uniform point inside each chosen triangle (formula above)
    raise NotImplementedError("sample points on the mesh surface")

elmo_pts = sample_points_on_mesh(elmo_vertices, elmo_triangles, n_points=20000)
print("sampled:", elmo_pts.shape)

### Check your sampler

Two automatic checks:

- **On the surface:** every sampled point should sit on Elmo's surface (within 0.0001 units). Points
  floating off the surface mean the step-3 formula is off.
- **Even coverage:** slice Elmo into 8 horizontal bands. Each band should get the same share of
  points as its share of surface area. A big mismatch means step 2 isn't weighting by area.

In [ ]:
assert elmo_pts.shape == (20000, 3), f"expected (20000, 3), got {elmo_pts.shape}"

# Check 1: distance from every sampled point to the mesh surface
scene = o3d.t.geometry.RaycastingScene()
scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(elmo_mesh))
dist_to_surface = scene.compute_distance(o3d.core.Tensor(elmo_pts.astype(np.float32))).numpy()
on_surface = np.mean(dist_to_surface < 1e-4)
print(f"points on the surface: {on_surface:.1%}")

# Check 2: share of points vs share of surface area in 8 height bands
corners = elmo_vertices[elmo_triangles]  # (T, 3, 3)
tri_area = 0.5 * np.linalg.norm(np.cross(corners[:, 1] - corners[:, 0], corners[:, 2] - corners[:, 0]), axis=1)
tri_height = corners[:, :, 1].mean(axis=1)
bands = np.quantile(tri_height, np.linspace(0, 1, 9))
area_share = np.histogram(tri_height, bands, weights=tri_area)[0] / tri_area.sum()
point_share = np.histogram(elmo_pts[:, 1], bands)[0] / len(elmo_pts)
worst_gap = np.abs(area_share - point_share).max()
print(f"worst band mismatch: {worst_gap:.1%} (should be under 3%)")

assert on_surface > 0.999, "Some points aren't on the surface -- check step 3"
assert worst_gap < 0.03, "Coverage is uneven -- check that step 2 weights triangles by area"
print("PASS - your sampler spreads points evenly over Elmo's surface.")

plot_clouds([elmo_pts], ["#ef4444"], ["Elmo (sampled)"], title=f"Elmo as a point cloud - {len(elmo_pts)} points", s=1)

# Save it -- this is now the same kind of data as the bunny scans in Parts 1-6.
o3d.io.write_point_cloud("data/kiri/elmo_points.ply",
                         o3d.geometry.PointCloud(o3d.utility.Vector3dVector(elmo_pts)))
print("saved data/kiri/elmo_points.ply")

### Stretch (optional, after Session 3): register two views of Elmo

Your Elmo point cloud is a complete object. Real scans aren't, so fake two partial scans and
fuse them with the pipeline you built (you need Part 4's `icp()` for this):

1. Make two overlapping "views": keep the points with `x < 0.3` as view A and `x > -0.3` as view B.
2. Rotate view B 10 degrees around its own center and shift it by about 0.05, keeping a copy of
   where it started.
3. Align view B back onto view A with your `icp()`, coarse-to-fine like Part 6: run it with
   `dist_threshold=0.2`, then 0.05, then 0.02. Measure how far the aligned points are from where
   view B started.

Then try narrowing the overlap to `x < 0.2` / `x > -0.2`. Alignment gets much worse. Why? This is
the same reason real scans need plenty of overlap between passes.